# Lecture 5 - Feed-forward networks in PyTorch

**ME324 - LSE Summer School**  |  Lab 5  |  2026-08-10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lse-me324/summer-school/blob/main/labs/lab-05-pytorch-feedforward.ipynb)

*(If the badge does not open, in Colab use **File -> Upload notebook** and pick this `.ipynb`.)*

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**And please use an LLM.** ChatGPT, Claude, Gemini, DeepSeek — whichever you like.
*"In PyTorch, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

---

### Where we are

You spent **Lab 3 and Lab 4** building automatic differentiation *by hand* with your
`Value` class, and you trained a tiny network with a hand-written update loop.
**From today, you never have to again.** PyTorch does the autodiff for you, runs on a
GPU when you have one, and lets you focus on *design* instead of bookkeeping.

In **Lab 2** you met a synthetic **voter-turnout** dataset and fit logistic regression to
it. Today we are going to move to PyTorch, and show how the Value class you built
previously can be written using an industry-standard language that simplifies some of
our manual work from before.

### Goals for today

By the end of this lab you will be able to:

1. See the exact parallel between your `Value` class and PyTorch tensors.
2. Define a feed-forward network **two ways** - an `nn.Module` subclass and `nn.Sequential` - and count its parameters.
3. Write the **canonical training loop** (`zero_grad -> forward -> loss -> backward -> step`) with `BCEWithLogitsLoss` and `Adam`.
4. Compare the net to a logistic-regression baseline and *see* the nonlinearity it captures.

> **Hardware:** this lab's dataset is tiny, so **CPU is completely fine** - you do *not*
> need a GPU. (In Colab: Runtime -> Change runtime type -> CPU is OK.)

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** building `TurnoutNet` (or `nn.Sequential`) and the PyTorch training loop.
- **Stretch / take-home — skip if short on time:** the sklearn baseline comparison and decision-boundary plot.

_Most of the code is written for you; the `# TODO` cells are the parts you write. Worked answers are in the **Solutions** section at the bottom._

## Run me first

Run this cell once to import everything and set the random seed. We fix
`torch.manual_seed(1337)` so your numbers match the notes (and your neighbour's).


In [ ]:
# === Run me first ===
# Colab already has these installed. If you run locally and something is missing:
#   pip install torch numpy matplotlib scikit-learn
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

# Reproducibility: the SAME seed we will use all course.
torch.manual_seed(1337)

print("torch       :", torch.__version__)
print("numpy       :", np.__version__)
print("CUDA (GPU)? :", torch.cuda.is_available(), "  <- False is totally fine for this lab")
print("\nSetup complete. This lab's data is tiny, so CPU runs in seconds.")


## The data: voter turnout (same as Lab 2)

This is the **same synthetic dataset you used by hand in Lab 2**. Each row is one
(simulated) voter with three features:

| feature | meaning | range |
|---|---|---|
| `age01` | age, rescaled to 0-1 | continuous in [0, 1] |
| `income01` | income, rescaled to 0-1 | continuous in [0, 1] |
| `voted2019` | did they vote last time? | 0 or 1 |

The label `y` is whether they **turned out** (1) or not (0).

The crucial detail (look at the generator): the true signal contains an
**interaction term**, `- 3.0 * age01 * income01`. That product term is a
**nonlinearity** - a straight-line model cannot represent it, but a network with a
hidden layer and a ReLU can. That is the whole point of today.


In [ ]:
# This is the SAME data-generating function you used by hand in Lab 2.
# Copy it verbatim -- do not change it.
def make_turnout_data(n=2000, seed=0):
    """Synthetic turnout data with a known NONLINEAR signal.
    Features: age01, income01, voted2019  (all in [0,1] or {0,1}).
    Label:    turned out (0/1).  Returns numpy arrays X (n,3), y (n,).
    The age*income interaction makes a single linear model imperfect,
    so a 2-layer net can beat plain logistic regression."""
    rng = np.random.default_rng(seed)
    age01     = rng.uniform(0, 1, n)
    income01  = rng.uniform(0, 1, n)
    voted2019 = rng.integers(0, 2, n).astype(float)
    # logit with an INTERACTION term (the nonlinearity) + a past-behaviour effect
    z = (1.6 * age01 + 1.2 * income01
         + 2.0 * voted2019
         - 3.0 * age01 * income01      # <-- interaction: the nonlinearity
         - 1.0)
    p = 1 / (1 + np.exp(-z))
    y = (rng.uniform(0, 1, n) < p).astype(np.int64)
    X = np.stack([age01, income01, voted2019], axis=1).astype(np.float32)
    return X, y

X, y = make_turnout_data(n=2000, seed=0)
print("X shape:", X.shape, "| y shape:", y.shape)
print("first 5 rows of X (age01, income01, voted2019):")
print(X[:5])
print("first 5 labels y:", y[:5])
print(f"fraction who turned out: {y.mean():.3f}")


### From numpy to tensors

PyTorch works with **tensors** (Lecture 5: "the building block"). A tensor is just a
multi-dimensional array that also knows how to track gradients. We convert our numpy
arrays to tensors and make a **train / validation split** so we can watch for
overfitting - exactly the "always" habit from the Lecture 5 checklist.


In [ ]:
# 1) Convert numpy -> torch tensors.
#    Features stay float32. Labels become float and get an extra dimension
#    (shape (n,1)) because BCEWithLogitsLoss compares against a column of logits.
#    NOTE: the lecture slide does this the other way round -- it leaves the labels
#    flat at (n,) and squeezes the logits with .squeeze(-1) inside the loop. Both
#    give the identical loss; the shapes just have to match EXACTLY. Shaping the
#    labels once here is a little tidier than squeezing on every single step.
X_t = torch.from_numpy(X)                       # (2000, 3) float32
y_t = torch.from_numpy(y).float().unsqueeze(1)  # (2000, 1) float32

# 2) Train / validation split (80 / 20). We shuffle indices with a fixed seed.
torch.manual_seed(1337)
n_total = X_t.shape[0]
n_train = int(0.8 * n_total)
perm    = torch.randperm(n_total)
train_idx, val_idx = perm[:n_train], perm[n_train:]

X_train, y_train = X_t[train_idx], y_t[train_idx]
X_val,   y_val   = X_t[val_idx],   y_t[val_idx]

print("X_train:", tuple(X_train.shape), "| y_train:", tuple(y_train.shape))
print("X_val  :", tuple(X_val.shape),   "| y_val  :", tuple(y_val.shape))
print("\nNote: age and income are already in [0,1] and voted2019 is 0/1,")
print("so our features are effectively normalised already (a good-practice box ticked).")


## Section 1 - `Value` -> PyTorch

Last week your `Value` class built a computation graph as you wrote an expression,
then `backward()` walked the graph filling in `.grad`. **PyTorch tensors do precisely
the same thing** - just faster, on arbitrary-shaped tensors, and battle-tested.

| You wrote in Lab 3/4 | PyTorch equivalent |
|---|---|
| `Value(2.0)` | `torch.tensor(2.0, requires_grad=True)` |
| building an expression records `_children` | building an expression records the graph |
| `.grad` on each node | `.grad` on each tensor |
| your `backward()` (topological sort + chain rule) | `.backward()` |
| your hand-written `p.data -= lr * p.grad` loop | `optimizer.step()` |

Run the cell below: it is the autograd example straight from the lecture.


In [ ]:
# Your Lab 4 Value class let you write an expression, call .backward(),
# and read .grad off the inputs. PyTorch tensors do EXACTLY the same thing.
x = torch.tensor(2.0, requires_grad=True)   # a leaf node we want gradients for
y_expr = x ** 3 + 4 * x                      # PyTorch builds the computation graph
y_expr.backward()                            # backprop from the scalar output

print("x.grad =", x.grad.item())            # d/dx (x^3 + 4x) = 3x^2 + 4 = 16 at x=2
print("Hand-check: 3 * 2**2 + 4 =", 3 * 2**2 + 4)


That is the **entire** autodiff API: build an expression from tensors with
`requires_grad=True`, call `.backward()` on the scalar output, read `.grad` off the
inputs. Behind the scenes is the exact algorithm you wrote yourselves.

From here on we let PyTorch own the gradients, and we build *models*.


## Section 2 - Define the model two ways

We will build the **same** network twice and prove they are identical:

* **(a) `nn.Module` subclass** - you write a `forward()` method. Flexible; use it when
  the forward pass branches or reuses layers. This matches the `TurnoutModel` from Lecture 5.
* **(b) `nn.Sequential`** - a plain stack of layers run in order. No `forward()` to
  write. Lecture 5 teaches this for straight stacks, and **today's lab uses it** for training.

Both are: `Linear(3 -> 16)  ->  ReLU  ->  Linear(16 -> 1)`.

A note on `nn.Linear(k, h)`: it is a fully-connected layer holding a `(h, k)` weight
matrix **and** an `h`-vector of biases, initialised for you (PyTorch uses a Kaiming-style
init - see Lecture 5). It computes `x @ W.T + b`.


In [ ]:
# (a) The nn.Module form. This matches the TurnoutModel from Lecture 5.
class TurnoutNet(nn.Module):
    def __init__(self, in_features=3, hidden=16):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)   # 3 -> 16
        self.fc2 = nn.Linear(hidden, 1)             # 16 -> 1

    def forward(self, x):
        h = torch.relu(self.fc1(x))   # hidden layer + ReLU activation
        # TODO: return the output layer (fc2) applied to h. One line.
        #       NOTE: we do NOT apply a sigmoid here -- the loss will (see Section 3).
        return ____                   # <-- replace ____

torch.manual_seed(1337)
print(TurnoutNet())


### (b) The same model as `nn.Sequential`

Same layers, same parameters - just less typing. Note that the activation `ReLU` is now
an explicit **layer object** `nn.ReLU()` rather than the function call `torch.relu(...)`.


In [ ]:
# (b) The nn.Sequential form: a plain stack of layers, run top to bottom.
#     This is EXACTLY the same network as TurnoutNet above.
torch.manual_seed(1337)
model_seq = nn.Sequential(
    nn.Linear(3, 16),   # 3 inputs -> 16 hidden
    nn.ReLU(),          # activation is now an explicit LAYER, not a function call
    nn.Linear(16, 1),   # 16 hidden -> 1 output (a logit)
)
print(model_seq)


### Same parameters - let's count them

Both forms hold **81** learnable numbers. Here is the arithmetic (don't forget the biases!):

$$
\underbrace{3 \times 16 + 16}_{\text{layer 1} \,=\, 64}
\;+\;
\underbrace{16 \times 1 + 1}_{\text{layer 2} \,=\, 17}
\;=\; \mathbf{81}\ \text{parameters.}
$$


In [ ]:
# Count the learnable parameters in BOTH models and check they match.
def count_params(m):
    return sum(p.numel() for p in m.parameters())

print("TurnoutNet params  :", count_params(TurnoutNet()))
print("nn.Sequential params:", count_params(model_seq))

print("\nPer-tensor breakdown of the Sequential model:")
for name, p in model_seq.named_parameters():
    print(f"  {name:14s} {tuple(p.shape)}  ->  {p.numel():>3d} numbers")

print("\nArithmetic, WITH biases:")
print("  layer 1 (Linear 3->16): 3*16 weights + 16 biases =", 3*16, "+", 16, "=", 3*16+16)
print("  layer 2 (Linear 16->1): 16*1 weights +  1 bias   =", 16*1, "+", 1,  "=", 16*1+1)
print("  TOTAL =", 3*16+16 + 16*1+1, "parameters")


## Section 3 - The training loop

To train our pytorch model, we need three things:

**1. The loss.** This is **binary classification**, and our model outputs a raw **logit**
(no sigmoid in `forward`). The right loss is **`nn.BCEWithLogitsLoss`**, which *bundles*
the sigmoid with binary cross-entropy in one numerically stable step.

> **Important:** do **not** put a sigmoid in your model's `forward` and *also* use a
> BCE-with-logits loss - you would apply the sigmoid twice. The rule (Lecture 5):
> the output activation lives **inside** the loss; you add it yourself only at *prediction*
> time. (`CrossEntropyLoss` works the same way for multi-class.)

**2. The optimiser.** `torch.optim.Adam` with learning rate `1e-3`...`1e-2` - the sensible
first try from Lecture 5.

**3. The loop.** Five steps, every iteration:

> **`zero_grad` -> `forward` -> `loss` -> `backward` -> `step`**

`zero_grad` first because PyTorch *accumulates* gradients - if you forget it, this step's
gradients pile on top of last step's. This exact pattern returns in **every** lab from here
to the GPT, so it is worth burning into memory now.


In [ ]:
# The loss and the optimiser. We use the SEQUENTIAL model from Section 2.
torch.manual_seed(1337)
model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))

loss_fn   = nn.BCEWithLogitsLoss()                       # sigmoid + BCE, fused & stable
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
print("loss:", loss_fn)
print("optimiser:", optimizer.__class__.__name__, "| lr =", optimizer.param_groups[0]["lr"])


### The loop itself

We use **full-batch** gradient descent: the dataset is small enough to use all of it each
step. (With big data you would iterate over mini-batches from a `DataLoader`; the five-step
body is identical - you will meet `DataLoader` in Lab 6.)


In [ ]:
# The canonical PyTorch training loop. Our data is tiny, so we use the whole
# training set each step (full-batch). With bigger data you would loop over
# mini-batches from a DataLoader -- you will see that from Lab 6 on.
#
# THIS FIVE-STEP PATTERN (zero_grad -> forward -> loss -> backward -> step)
# recurs in EVERY training loop for the rest of the course. Learn it once.
def train(model, loss_fn, optimizer, epochs=2000):
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        model.train()
        ____                                # TODO 1: clear the old gradients
        logits = ____                       # TODO 2: forward pass of model on X_train
        loss   = loss_fn(logits, y_train)   # (given) compute the loss
        ____                                # TODO 3: backpropagate (fill in .grad)
        ____                                # TODO 4: one optimiser step (update weights)

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val)
        history["train"].append(loss.item())
        history["val"].append(val_loss.item())
    return history

history = train(model, loss_fn, optimizer, epochs=2000)
print("done. final val loss:", round(history["val"][-1], 4))


### Read the curves

Train and validation loss should both fall and then flatten. If the **val** curve started
rising while **train** kept falling, that gap would be **overfitting** (Lecture 5, Part 3).
On this small, noisy problem the two curves stay close - we are not overfitting.


In [ ]:
# Plot the training and validation loss curves.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train"], label="train")
ax.plot(history["val"],   label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("BCE loss (lower = better)")
ax.set_title("Training curve")
ax.legend()
plt.show()

print(f"final train loss: {history['train'][-1]:.4f}")
print(f"final val   loss: {history['val'][-1]:.4f}")


In [ ]:
# Turnout accuracy of the trained net on train and val sets.
@torch.no_grad()
def net_probs(model, X):
    model.eval()
    return torch.sigmoid(model(X))   # logits -> probabilities (we add sigmoid HERE, for prediction)

train_acc = ((net_probs(model, X_train) > 0.5).float() == y_train).float().mean().item()
val_acc   = ((net_probs(model, X_val)   > 0.5).float() == y_val).float().mean().item()
print(f"net  train accuracy: {train_acc:.3f}")
print(f"net  val   accuracy: {val_acc:.3f}")


## Section 4 - Compare to baselines

Does the network actually buy us anything over the **logistic regression** from Lab 2
(and the by-hand model from Lab 4)? Logistic regression is a single linear layer + sigmoid -
it can only draw a **straight** decision boundary. Our net has a hidden layer and a ReLU, so
it can **bend**. Let's measure, then look.


In [ ]:
# Baseline 1: logistic regression (the Lab 2 / sklearn model) on the SAME features.
Xtr_np, ytr_np = X_train.numpy(), y_train.numpy().ravel()
Xva_np, yva_np = X_val.numpy(),   y_val.numpy().ravel()

logreg = LogisticRegression(max_iter=2000)
logreg.fit(Xtr_np, ytr_np)

# Probabilities for fair comparison (log-loss is the same BCE the net minimised).
net_p_val    = net_probs(model, X_val).numpy().ravel()
logreg_p_val = logreg.predict_proba(Xva_np)[:, 1]

# A "cheat" reference: the TRUE probability from the data generator (the best any
# model could possibly do). The gap to this is IRREDUCIBLE noise, not model error.
z_true = (1.6*Xva_np[:,0] + 1.2*Xva_np[:,1] + 2.0*Xva_np[:,2]
          - 3.0*Xva_np[:,0]*Xva_np[:,1] - 1.0)
p_true = 1 / (1 + np.exp(-z_true))

print(f"{'model':<26}{'val accuracy':>14}{'val log-loss':>14}")
print(f"{'logistic regression':<26}{accuracy_score(yva_np, logreg_p_val>0.5):>14.3f}"
      f"{log_loss(yva_np, logreg_p_val):>14.4f}")
print(f"{'neural net':<26}{accuracy_score(yva_np, net_p_val>0.5):>14.3f}"
      f"{log_loss(yva_np, net_p_val):>14.4f}")
print(f"{'BAYES-OPTIMAL (true p)':<26}{accuracy_score(yva_np, p_true>0.5):>14.3f}"
      f"{log_loss(yva_np, p_true):>14.4f}")


### Reading the table

Two honest lessons here:

* **The net's log-loss beats logistic regression's** and sits right next to the
  "Bayes-optimal" row (the best *any* model could do, computed from the true generating
  probabilities). The net has learned the age*income interaction that the linear model
  structurally cannot represent.
* **Accuracy barely moves**, because this data is *genuinely noisy*: even the
  Bayes-optimal model only reaches about 0.72-0.74. That ceiling is **irreducible error** -
  not a flaw in your model. (This is why we also report log-loss, which is more sensitive
  to the *quality of the probabilities*, not just the hard 0/1 call.)


### See the nonlinearity

Below, each panel shows a model's predicted `P(turnout)` across the age-income square,
holding `voted2019 = 0`, with the real data points overlaid.

* The **neural net** (left) bends: it pushes probability up in the corners where age and
  income disagree, and down where they are both high or both low - this *is* the
  `- 3.0 * age*income` interaction, learned from data.
* **Logistic regression** (right) can only produce **flat, parallel bands** - it has no way
  to express that interaction, so it averages over it and misses the structure.

That curved-vs-flat picture is the entire reason we reach for neural networks on data with
interactions like this.


In [ ]:
# Visualise WHY the net wins: plot each model's predicted P(turnout) over a grid
# of (age, income), holding voted2019 = 0. The net can BEND; logistic regression
# can only draw flat, parallel bands.
gx = np.linspace(0, 1, 200)
GX, GY = np.meshgrid(gx, gx)
levels = np.linspace(0, 1, 21)
voted_slice = 0
grid = np.stack([GX.ravel(), GY.ravel(),
                 np.full(GX.size, float(voted_slice))], axis=1).astype(np.float32)

with torch.no_grad():
    model.eval()
    net_surface = torch.sigmoid(model(torch.from_numpy(grid))).numpy().reshape(GX.shape)
logreg_surface = logreg.predict_proba(grid)[:, 1].reshape(GX.shape)

mask = (X[:, 2] == voted_slice)   # data points in this slice, to overlay
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
for ax, (surface, title) in zip(
        axes, [(net_surface, "Neural net (curved)"),
               (logreg_surface, "Logistic regression (flat planes)")]):
    cf = ax.contourf(GX, GY, surface, levels=levels, cmap="RdBu_r", vmin=0, vmax=1)
    ax.contour(GX, GY, surface, levels=[0.5], colors="black", linewidths=3)
    ax.scatter(X[mask, 0], X[mask, 1], c=y[mask], cmap="RdBu_r",
               edgecolors="k", s=14, linewidths=0.3, vmin=0, vmax=1)
    ax.set_xlabel("age (normalised)")
    ax.set_ylabel("income (normalised)")
    ax.set_title(title)
    fig.colorbar(cf, ax=ax, label="P(turnout)")
fig.suptitle("Predicted turnout surface  (voted2019 = 0 slice)")
fig.tight_layout()
plt.show()


## Recap

You just rebuilt your Lab 4 network in PyTorch and trained it on the Lab 2 data. You can now:

* read tensors and autograd as the grown-up version of your `Value` class;
* define a feed-forward net as an `nn.Module` **or** `nn.Sequential`, and count its parameters;
* write the canonical loop: **`zero_grad -> forward -> loss -> backward -> step`**;
* pick `BCEWithLogitsLoss` + `Adam`, track train/val loss, and compare against a baseline;
* *see* the nonlinearity a hidden layer + ReLU captures.

**You can now train (more or less) any feed-forward network in PyTorch.**

### Extensions (optional)

* Swap `nn.ReLU()` for `nn.Sigmoid()` and watch the loss curve - does it train slower?
* Bump the learning rate to `1e-1`. Does the loss get noisy or explode?
* Add a second hidden layer (`Linear(16,16) -> ReLU`). Does val loss improve or overfit?

### Next time - Lab 6: images and convolutions

Same training loop, **new layers**. We swap the tabular turnout data for *images*
(FashionMNIST) and the `Linear` layers for **convolutions** - but
`zero_grad -> forward -> loss -> backward -> step` stays exactly the same. Once you have
that loop, the rest of the course is mostly "what goes inside the model".


## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — the forward pass**

In [ ]:
class TurnoutNet(nn.Module):
    def __init__(self, in_features=3, hidden=16):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h)            # output layer -> a single logit per row

torch.manual_seed(1337)
print(TurnoutNet())

**Solution — the training loop**

In [ ]:
# Self-contained: re-creates a fresh model/loss/optimiser so this cell reproduces
# the intended result on its own, regardless of what ran before it.
torch.manual_seed(1337)
model     = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))
loss_fn   = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

def train(model, loss_fn, optimizer, epochs=2000):
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()               # 1. clear the old gradients
        logits = model(X_train)             # 2. forward pass
        loss   = loss_fn(logits, y_train)   #    compute the loss
        loss.backward()                     # 3. backpropagate
        optimizer.step()                    # 4. update the weights

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val)
        history["train"].append(loss.item())
        history["val"].append(val_loss.item())
    return history

history = train(model, loss_fn, optimizer, epochs=2000)
print("done. final val loss:", round(history["val"][-1], 4))